In [1]:
# Imports section
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures

## Part 1. Loading the dataset

In [2]:
# Using pandas load the dataset (load remotely, not locally)
# Output the first 15 rows of the data
# Display a summary of the table information (number of datapoints, etc.)

df = pd.read_csv('science_data_large.csv')

In [3]:
df.head(15)

,Temperature °C,Mols KCL,Size nm^3
0,469,647,6.244743e+05
1,403,694,5.779610e+05
2,302,975,6.196847e+05
3,779,916,1.460449e+06
4,901,18,4.325726e+04
5,545,637,7.124634e+05
6,660,519,7.006960e+05
7,143,869,2.718260e+05
8,89,461,8.919803e+04
9,294,776,4.770210e+05


In [4]:
df.keys()

Index(['Temperature °C', 'Mols KCL', 'Size nm^3'], dtype='object')

## Part 2. Splitting the dataset

In [5]:
# Take the pandas dataset and split it into our features (X) and label (y)
X = df[['Temperature °C', 'Mols KCL']]
y = df['Size nm^3']
# Use sklearn to split the features and labels into a training/test set. (90% train, 10% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1)

## Part 3. Perform a Linear Regression

In [6]:
# Use sklearn to train a model on the training set
reg = LinearRegression().fit(X, y)

# Create a sample datapoint and predict the output of that sample with the trained model
datapoint = pd.DataFrame({'Temperature °C': 468, 'Mols KCL': 646}, index=[0])
print(f'Data point: \n{datapoint}')
print(f'Data point prediction: {reg.predict(datapoint)[0]:.3E}\n')

# Report on the score for that model, in your own words (markdown, not code) explain what the score means
print(f'Model score: {reg.score(X_test, y_test)}\n')

# Extract the coefficents and intercept from the model and write an equation for your h(x) using LaTeX
print(f'Equation: h(x) = {reg.intercept_:.3E} + {reg.coef_[0]:.3E}t + {reg.coef_[1]:.3E}m')

Data point: 
   Temperature °C  Mols KCL
0             468       646
Data point prediction: 6.601E+05

Model score: 0.8734553853493363

Equation: h(x) = -4.162E+05 + 8.759E+02t + 1.032E+03m


In [7]:
coefficients = pd.DataFrame({"Feature":X.columns,"Coefficients":np.transpose(reg.coef_)})
coefficients

,Feature,Coefficients
0,Temperature °C,875.909927
1,Mols KCL,1031.595025


**Score**: It is an evaluation metric of how optimal the linear regression model is performing by applying the model on the test set. A score of 1 would suggest that the model works perfectly on the test set, and a lower score (possibly negative) would suggest a bad performance.

**Equation:**

$$
h(x) = -4.162\cdot10^5 + 8.759\cdot10^2t + 1.032\cdot10^3m
$$

where: 

$$
t \rightarrow Temperature (°C)
\\m \rightarrow Mols (KCL)
$$

## Part 4. Use Cross Validation

In [8]:
# Use the cross_val_score function to repeat your experiment across many shuffles of the data
score = cross_val_score(reg, X_train,y_train)

# Report on their finding and their significance
print(f'Cross Validation Score: {score}')
print(f'Average Score: {score.mean():.4f}')
print(f'Variance: {score.var():.4f}')

Cross Validation Score: [0.86078302 0.86047883 0.85690824 0.85060243 0.85170574]
Average Score: 0.8561
Variance: 0.0000


With an average score of 0.8627, this shows that the model is doing well in every fold of the data. The low variance suggests that the model is also performing consistently on every fold of data. 

## Part 5. Using Polynomial Regression

In [9]:
# Using the PolynomialFeatures library perform another regression on an augmented dataset of degree 2
poly = PolynomialFeatures(2, include_bias=False)


X_train_poly = poly.fit_transform(X_train)
X_test_poly = poly.fit_transform(X_test)

feature_names = poly.get_feature_names_out()

reg_poly = LinearRegression().fit(X_train_poly, y_train)

# Report on the metrics and output the resultant equation as you did in Part 3.
print(f'Model score: {reg_poly.score(X_test_poly, y_test)}\n')
print(f'Bias: {reg_poly.intercept_:.3E}')
for i in range(5):
    print(f'{feature_names[i]}: {reg_poly.coef_[i]:.3f}')

Model score: 1.0

Bias: 1.661E-05
Temperature °C: 12.000
Mols KCL: -0.000
Temperature °C^2: 0.000
Temperature °C Mols KCL: 2.000
Mols KCL^2: 0.029


Equation:

$$
h(x)=2.053\cdot10^{-5}+12t+2tm+0.029m^2
$$

where

$$
t\rightarrow Temperature (°C)
\\ m\rightarrow Mols (KCL)
$$